<a id="top"></a>

# Demo: the loop made visible (on a stub model)

```yaml
title: "Demo: the loop made visible (on a stub model)"
keywords: agent loop, run_loop, stub model, FakeModel, natural termination, max_steps cap, tool failure, tool_result, is_error, offline, teacher demo
estimated duration: 18
```

> **Lesson:** L10. Teacher demo — the loop concept from the [roadmap](../../../../docs/origin/lesson_roadmaps/L10/demos_or_activities.md). **No API key needed.** This demo drives the loop with a *scripted stub model* so termination, the `max_steps` cap, and tool-failure handling are **deterministic** — the same run every time. The live multi-step version is [L1006](L1006_lecture.ipynb).
>
> The point to land: **an agent is a loop, not a model.** The model is a stateless function call; the loop calls it, runs the tool it asked for, feeds the result back, and repeats — until the model stops asking (natural termination) or a cap you chose fires.

## Contents

- [1. Setup — tools, a stub model, and the loop](#1-setup--tools-a-stub-model-and-the-loop)
- [2. The loop itself](#2-the-loop-itself)
- [3. Natural termination — the happy path](#3-natural-termination--the-happy-path)
- [4. The max_steps cap — catching a runaway](#4-the-max_steps-cap--catching-a-runaway)
- [5. Tool failure as a message, not a crash](#5-tool-failure-as-a-message-not-a-crash)
- [6. Takeaways](#6-takeaways)

## 1. Setup — tools, a stub model, and the loop

Three pieces. (1) Two inline tools — `calculator` and `lookup` — and a **dispatch table** mapping each name to its function. (2) A `FakeModel` whose `.create(...)` pops the next *scripted* response off a list; it mimics the SDK's `.content` blocks with `SimpleNamespace`, so the loop can't tell it from a real client. (3) `dispatch`, which runs one tool and — **Rule 3** — turns any exception into a `tool_result` with `is_error=True`.

In [1]:
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the whitelist above blocks names/attributes/calls.
    return str(eval(expression))


POPULATIONS = {
    "tokyo": "37000000",
    "lagos": "15000000",
    "paris": "11000000",
}


def lookup(city: str) -> str:
    """Return a city's population from a tiny in-memory table, or raise KeyError if missing."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# The dispatch table: tool NAME -> the Python function that runs it. The loop
# never hard-codes tool names; it looks each requested name up here.
TOOLS: dict[str, object] = {"calculator": calculator, "lookup": lookup}

In [2]:
from types import SimpleNamespace


def text_block(s: str) -> SimpleNamespace:
    """A fake assistant text block (mimics the SDK's text content block)."""
    return SimpleNamespace(type="text", text=s)


def tool_use_block(call_id: str, name: str, args: dict[str, object]) -> SimpleNamespace:
    """A fake assistant tool_use block (mimics the SDK's tool_use content block)."""
    return SimpleNamespace(type="tool_use", id=call_id, name=name, input=args)


def response(blocks: list[SimpleNamespace]) -> SimpleNamespace:
    """A fake model response. stop_reason is 'tool_use' if any block is a tool_use."""
    stop = "tool_use" if any(b.type == "tool_use" for b in blocks) else "end_turn"
    return SimpleNamespace(content=blocks, stop_reason=stop)


class FakeModel:
    """A scripted stand-in for the Anthropic client.

    Its .create(...) ignores the messages and simply pops the NEXT pre-scripted
    response off a list. This makes the loop fully deterministic: no API key, no
    cost, and the exact same run every time -- ideal for exercising control flow.
    If the script runs out (the loop called more times than scripted), it keeps
    returning the same final tool_use, which is how we simulate a RUNAWAY model.
    """

    def __init__(self, scripted: list[SimpleNamespace]) -> None:
        self._scripted = scripted
        self.calls = 0

    def create(self, **kwargs: object) -> SimpleNamespace:
        # kwargs (messages=, tools=, model=, max_tokens=) are accepted and ignored;
        # a real client would use them. The stub just returns the next script line.
        i = min(self.calls, len(self._scripted) - 1)
        self.calls += 1
        return self._scripted[i]

In [3]:
def dispatch(tools: dict[str, object], call: object) -> dict[str, object]:
    """Run one requested tool and return a tool_result block.

    On success: a tool_result carrying the tool's output. On ANY exception (unknown
    tool name, the tool raised): a tool_result with is_error=True and a SHORT message
    -- never a traceback. The loop keeps going; the model decides what to do next.
    """
    fn = tools.get(call.name)
    if fn is None:
        return {
            "type": "tool_result",
            "tool_use_id": call.id,
            "content": f"no such tool {call.name!r}",
            "is_error": True,
        }
    try:
        output = fn(**call.input)
        return {"type": "tool_result", "tool_use_id": call.id, "content": output}
    except Exception as exc:
        # repr(exc) is a class name + one line: e.g. KeyError('no population...').
        # Short, descriptive, cheap -- never dump a traceback at the model.
        return {
            "type": "tool_result",
            "tool_use_id": call.id,
            "content": repr(exc),
            "is_error": True,
        }

[↑ Back to top](#top)

## 2. The loop itself

`run_loop(model, tools, user_msg, max_steps)`. Each iteration: call the model; if it emitted `tool_use` blocks, run **every** one and package **all** the `tool_result`s into a **single** user-role message (**Rule 1** — the message-history invariant), then loop; if it emitted only text, return it. The `max_steps` cap forces a halt even if the model still wants tools.

In [4]:
from dataclasses import dataclass


@dataclass
class RunResult:
    """What the loop returns: the answer, how many model calls it took, and WHY it stopped."""

    final_text: str
    iterations: int
    termination: str  # "natural" or "max_steps"


def run_loop(
    model: object,
    tools: dict[str, object],
    user_msg: str,
    max_steps: int,
) -> RunResult:
    """Run a model -> tool -> model loop until the model stops asking for tools.

    Each iteration: call the model; if it emitted tool_use blocks, run EVERY one,
    package ALL their tool_results into a single user-role message, and loop; if it
    emitted only text, return it (natural termination). The max_steps cap forces a
    halt even if the model still wants tools -- termination is a design decision.

    Tool failures are turned into messages, not crashes: a tool that raises becomes a
    tool_result with is_error=True, handed back so the model can decide what to do.
    """
    messages: list[dict[str, object]] = [{"role": "user", "content": user_msg}]

    for step in range(1, max_steps + 1):
        resp = model.create(
            model="claude-sonnet-4-6",
            max_tokens=512,
            tools=list(tools),
            messages=messages,
        )
        tool_uses = [b for b in resp.content if b.type == "tool_use"]

        # No tool_use block -> the model thinks it's done. NATURAL termination.
        if not tool_uses:
            final_text = "".join(b.text for b in resp.content if b.type == "text")
            print(f"[step {step}] natural termination (no tool_use)")
            return RunResult(final_text=final_text, iterations=step, termination="natural")

        # Rule 1: record the assistant turn, then answer EVERY tool_use with a
        # matching tool_result in ONE user-role message before the next call.
        messages.append({"role": "assistant", "content": resp.content})
        results: list[dict[str, object]] = []
        for call in tool_uses:
            print(f"[step {step}] tool_use: {call.name}({call.input})")
            results.append(dispatch(tools, call))
        messages.append({"role": "user", "content": results})

    # Fell out of the for-loop: the cap fired before the model finished.
    print(f"[stop] max_steps={max_steps} hit -- non-natural termination")
    return RunResult(final_text="", iterations=max_steps, termination="max_steps")

[↑ Back to top](#top)

## 3. Natural termination — the happy path

Script a model that calls `calculator`, then (seeing the result) calls `lookup`, then answers in plain text. Two tool calls, then a final message with **no `tool_use`** — that is natural termination. Watch the per-step prints walk the loop.

In [5]:
happy_script = [
    response([tool_use_block("c1", "calculator", {"expression": "288 * 0 + 1"})]),
    response([tool_use_block("c2", "lookup", {"city": "Tokyo"})]),
    response([text_block("Tokyo's population is about 37,000,000.")]),
]
model = FakeModel(happy_script)
result = run_loop(model, TOOLS, "What is the population of Tokyo?", max_steps=10)
print()
print("final_text :", result.final_text)
print("iterations :", result.iterations)
print("termination:", result.termination)

[step 1] tool_use: calculator({'expression': '288 * 0 + 1'})
[step 2] tool_use: lookup({'city': 'Tokyo'})
[step 3] natural termination (no tool_use)

final_text : Tokyo's population is about 37,000,000.
iterations : 3
termination: natural


[↑ Back to top](#top)

## 4. The max_steps cap — catching a runaway

Now a model that **never stops**: it keeps asking for the same `lookup` call. With a real model this is the classic runaway. The `FakeModel` reuses its last script line forever, so the loop would spin without a cap. `max_steps=4` catches it — and **hitting the cap is always a signal worth investigating**, not noise.

In [6]:
runaway_script = [
    response([tool_use_block("r1", "lookup", {"city": "Tokyo"})]),
]
model = FakeModel(runaway_script)
result = run_loop(model, TOOLS, "Keep checking Tokyo's population.", max_steps=4)
print()
print("termination:", result.termination, "  <- the cap fired, not the model")
print("iterations :", result.iterations)
assert result.termination == "max_steps"

[step 1] tool_use: lookup({'city': 'Tokyo'})
[step 2] tool_use: lookup({'city': 'Tokyo'})
[step 3] tool_use: lookup({'city': 'Tokyo'})
[step 4] tool_use: lookup({'city': 'Tokyo'})
[stop] max_steps=4 hit -- non-natural termination

termination: max_steps   <- the cap fired, not the model
iterations : 4


[↑ Back to top](#top)

## 5. Tool failure as a message, not a crash

Script the model to look up a city **not in the table** (`lookup` raises `KeyError`), then recover by looking up one that *is*. Without `dispatch`'s `try/except` the `KeyError` would crash the whole agent. Instead the loop converts it to a `tool_result` with `is_error=True`, the model 'sees' the error on its next turn, and finishes gracefully.

In [7]:
recover_script = [
    response([tool_use_block("f1", "lookup", {"city": "Atlantis"})]),  # not in table -> KeyError
    response([tool_use_block("f2", "lookup", {"city": "Paris"})]),  # recover
    response([text_block("I couldn't find Atlantis, but Paris is about 11,000,000.")]),
]
model = FakeModel(recover_script)
result = run_loop(model, TOOLS, "Population of Atlantis? If unknown, try Paris.", max_steps=10)
print()
print("final_text :", result.final_text)
print("termination:", result.termination)
# Show the error tool_result the loop fed back after the first (failing) call:
bad_call = tool_use_block("f1", "lookup", {"city": "Atlantis"})
print("error tool_result handed to the model:", dispatch(TOOLS, bad_call))

[step 1] tool_use: lookup({'city': 'Atlantis'})
[step 2] tool_use: lookup({'city': 'Paris'})
[step 3] natural termination (no tool_use)

final_text : I couldn't find Atlantis, but Paris is about 11,000,000.
termination: natural
error tool_result handed to the model: {'type': 'tool_result', 'tool_use_id': 'f1', 'content': 'KeyError("no population on file for \'Atlantis\'")', 'is_error': True}


[↑ Back to top](#top)

## 6. Takeaways

- **An agent is a loop, not a model.** The same `run_loop` drove all three runs above; only the *script* changed. (In [L1006](L1006_lecture.ipynb) only the *client* changes — a real model replaces the stub, loop unchanged.)
- **Termination is a design decision.** Natural termination = the model returned plain text. The `max_steps` cap is a *safety net* for everything else; a loop with no cap is broken, not minimal.
- **Tool failures are messages, not exceptions.** `dispatch` turns a raise into a `tool_result(is_error=True)` with a short `repr(exc)` — never a traceback — and the model decides whether to retry (reinforces [L08](../L08/objectives.md)'s error thinking at the *loop* layer).
- **A stub model makes control flow testable.** Scripting the model is how you exercise termination and failure *offline and reproducibly* — the same mocking stance as the project's tests.
- Next: practice the loop in [L1004](L1004_lab_empty.ipynb), failures in [L1005](L1005_lab_empty.ipynb), then watch it drive a real model in [L1006](L1006_lecture.ipynb).

[↑ Back to top](#top)